# 4 · Combine searches and measure what worked

One searcher is good at words; another is good at visual clues. Strategy C asks both,
then combines their ranked results. We will implement **reciprocal rank fusion (RRF)**,
expose a duplicate-source trap, and evaluate retrieval with carefully named metrics.

Allow **45–65 minutes**. Our default experiment uses a lexical memo searcher and the
color-only image searcher. Optional FastEmbed experiments compare real A/B/C on the
same labelled questions. The default scores describe the teaching baselines only.

In [1]:
import os, sys, json, copy
from pathlib import Path
import numpy as np
from IPython.display import display, Image, Markdown
from dotenv import load_dotenv, find_dotenv

# Works when Jupyter starts either here or at the repository root.
candidates = [Path.cwd(), Path.cwd() / "evals" / "multimodal"]
MODULE = next((p for p in candidates if (p / "lab.py").exists()), None)
if MODULE is None:
    raise RuntimeError("Start Jupyter from evals/multimodal or the repository root.")
os.chdir(MODULE)
if str(MODULE) not in sys.path:
    sys.path.insert(0, str(MODULE))
load_dotenv(find_dotenv())  # Secrets live outside the notebook.
from lab import (
    CORPUS, GOLD, Record, LexicalEncoder, ColorEncoder, QdrantIndex,
    load_corpus, load_reference_parses, normalize, validate_parse,
    flatten_parse, build_strategy_a, toy_shared_space, show_hits, rrf,
    evaluate, check_data_points, feature_fingerprint, read_pdf,
    sample_keyframes, build_video_records, timestamp, evidence_bundle,
)
RUN_MODELS = os.getenv("MM_RUN_MODELS") == "1"
RUN_LIVE = os.getenv("MM_RUN_LIVE") == "1"
print("Local pretrained models enabled:", RUN_MODELS)
print("Remote API calls enabled:", RUN_LIVE)
print("Default path: explicit lexical/color baselines, real in-memory Qdrant.")

Local pretrained models enabled: False
Remote API calls enabled: False
Default path: explicit lexical/color baselines, real in-memory Qdrant.


/Users/pmui/dev/agents2026/evals/multimodal/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


![Separate text and image searches feed one rank-fusion stage.](assets/06_strategy_c.svg)

## Step 1 · Why not take the largest raw score?

Imagine a text searcher returns scores `0.42, 0.39` and an image searcher returns
`0.81, 0.80`. The image results are not automatically better. Their scores are produced
by different models with different geometries and distributions. Even cosine scores
sharing the interval [-1, 1] are not calibrated relevance probabilities.

RRF uses **positions**, not raw similarities. For a source at rank `r`, it adds
`1 / (c + r)`. We use one-based ranks. `c=60` is a common starting setting, not a law.
Larger `c` makes nearby ranks contribute more similarly. It does not create relevance
when both input rankings are poor.

![Worked reciprocal-rank contributions combine evidence from multiple ranked lists.](assets/07_rrf.svg)

In [2]:
# Worked miniature: source B occurs in both lists.
def toy_hit(identifier, score):
    return {"id": identifier, "source_id": identifier, "modality": "illustration", "score": score}
first = [toy_hit("A", 0.42), toy_hit("B", 0.39)]
second = [toy_hit("B", 0.81), toy_hit("C", 0.80)]
worked_fusion = rrf([first, second], constant=60)
display(show_hits(worked_fusion))
print("B's expected fused score:", 1/62 + 1/61)
print("B's observed contributions:", worked_fusion[0]["votes"])

[{'rank': 1,
  'id': 'B',
  'source': 'B',
  'modality': 'illustration',
  'score': 0.0325},
 {'rank': 2,
  'id': 'A',
  'source': 'A',
  'modality': 'illustration',
  'score': 0.0164},
 {'rank': 3,
  'id': 'C',
  'source': 'C',
  'modality': 'illustration',
  'score': 0.0161}]

B's expected fused score: 0.03252247488101534
B's observed contributions: [{'list': 0, 'rank': 2, 'contribution': 0.016129032258064516}, {'list': 1, 'rank': 1, 'contribution': 0.01639344262295082}]


### Open RRF: a score accumulator

Here is the algorithm without a helper. Walk each ranked list, convert the one-based
rank into a contribution, and add it to that source's total. Sort the totals after
both lists have voted. This short version is valid for these lists because every
source appears at most once in each list; the next section handles duplicate chunks.

In [3]:
totals = {}
for ranked_list in [first, second]:
    for rank, hit in enumerate(ranked_list, start=1):
        source = hit["source_id"]
        contribution = 1.0 / (60 + rank)
        totals[source] = totals.get(source, 0.0) + contribution
manual_order = sorted(totals, key=lambda source: (-totals[source], source))
print("Manual totals:", totals)
print("Manual order:", manual_order)
assert manual_order == [h["source_id"] for h in worked_fusion]
assert all(np.isclose(totals[h["source_id"]], h["score"]) for h in worked_fusion)

Manual totals: {'A': 0.01639344262295082, 'B': 0.03252247488101534, 'C': 0.016129032258064516}
Manual order: ['B', 'A', 'C']


B wins because both lists support it. A is first in its list and C is second, so A
contributes `1/61` and C contributes `1/62`; they do **not** tie. This difference helps us
catch an easy off-by-one mistake before using a large dataset.

If one list were duplicated ten times, its candidates would receive ten votes. RRF
assumes the chosen list sources are meaningful; it does not know whether searchers
are independent or whether one source is being counted repeatedly.

## Step 2 · Give chunks and sources different identities

One PDF can have many pages; one video segment can have several sampled frames.
Each indexed chunk needs a unique point ID. But fusion may need one vote per source
per list. For this notebook, an image or memo is a source. For video, a timed segment
will be a source and the complete video its parent.

The next example gives A two chunks in the same list. Our RRF deduplicates at the
source level **before assigning compact ranks**: A gets one vote, B gets rank 2.
Distinct chunk evidence is retained for inspection. Ties use stable source IDs.

In [4]:
duplicate_chunks = [
    {**toy_hit("A-frame-0", .9), "source_id": "A"},
    {**toy_hit("A-frame-1", .8), "source_id": "A"},
    toy_hit("B", .7),
]
deduped = rrf([duplicate_chunks])
display(show_hits(deduped))
print("Votes:", {h["source_id"]: h["votes"] for h in deduped})
assert len(deduped[0]["votes"]) == 1
assert len(deduped[0]["evidence"]) == 2

[{'rank': 1,
  'id': 'A-frame-0',
  'source': 'A',
  'modality': 'illustration',
  'score': 0.0164},
 {'rank': 2,
  'id': 'B',
  'source': 'B',
  'modality': 'illustration',
  'score': 0.0161}]

Votes: {'A': [{'list': 0, 'rank': 1, 'contribution': 0.01639344262295082}], 'B': [{'list': 0, 'rank': 2, 'contribution': 0.016129032258064516}]}


## Step 3 · Build two independent stores

The text branch below indexes **only the three raw memos**. The image branch indexes
the four images with measured color vectors. This mirrors the separate-store design;
no caption-derived text is quietly added to C's memo branch.

A, in contrast, indexes memos **and hand-authored image descriptions**. We must report
that extra information when comparing strategies. A better score for A may partly
reflect better input descriptions, not an intrinsically better retrieval architecture.

In [5]:
raw = load_corpus()
memos = [r for r in raw if r.modality == "text"]
text_encoder = LexicalEncoder([r.text for r in memos])
text_index = QdrantIndex(memos, text_encoder.embed([r.text for r in memos]), "c_memos")
images, color_encoder, image_index = toy_shared_space(raw)
_, a_encoder, a_index = build_strategy_a(raw)

def retrieve_a(query, k=3):
    return a_index.search(a_encoder.embed([query])[0], k)

def retrieve_b_color(query, k=3):
    return image_index.search(color_encoder.embed_text([query])[0], k)

def retrieve_c(query, k=3):
    text_hits = text_index.search(text_encoder.embed([query])[0], k)
    image_hits = retrieve_b_color(query, k)
    return rrf([text_hits, image_hits])[:k]

query = "green water cleaning methods"
print("Text branch:")
display(show_hits(text_index.search(text_encoder.embed([query])[0], 3)))
print("Image branch:")
display(show_hits(retrieve_b_color(query)))
print("Fused:")
display(show_hits(retrieve_c(query)))

Text branch:


[{'rank': 1,
  'id': 'memo_cleaning',
  'source': 'memo_cleaning',
  'modality': 'text',
  'score': 0.5714},
 {'rank': 2,
  'id': 'memo_safety',
  'source': 'memo_safety',
  'modality': 'text',
  'score': 0.0},
 {'rank': 3,
  'id': 'memo_schedule',
  'source': 'memo_schedule',
  'modality': 'text',
  'score': 0.0}]

Image branch:


[{'rank': 1,
  'id': 'chart_water',
  'source': 'chart_water',
  'modality': 'image',
  'score': 1.0},
 {'rank': 2,
  'id': 'chart_temperature',
  'source': 'chart_temperature',
  'modality': 'image',
  'score': 0.0212},
 {'rank': 3,
  'id': 'chart_energy',
  'source': 'chart_energy',
  'modality': 'image',
  'score': 0.0184}]

Fused:


[{'rank': 1,
  'id': 'chart_water',
  'source': 'chart_water',
  'modality': 'image',
  'score': 0.0164},
 {'rank': 2,
  'id': 'memo_cleaning',
  'source': 'memo_cleaning',
  'modality': 'text',
  'score': 0.0164},
 {'rank': 3,
  'id': 'chart_temperature',
  'source': 'chart_temperature',
  'modality': 'image',
  'score': 0.0161}]

The image branch can use “green”; the memo branch can use “water” and “cleaning.”
Here C requests `k` candidates **per branch**, then returns a final `k`. It examines
up to `2k` candidates, while A requests only `k`. A matched final evidence count is
not a matched candidate or compute budget. Report both budgets in any comparison.

Neither query branch depends on the other's result. A service can execute independent
branches concurrently and wait for both before fusion. Our normal function is
sequential so intermediate values stay easy to inspect.

The next cell demonstrates concurrent orchestration using separate local stores.
Tiny local operations may not get faster because thread overhead can dominate.
Concurrency is a scheduling choice, not a retrieval-quality improvement.

In [6]:
from concurrent.futures import ThreadPoolExecutor
with ThreadPoolExecutor(max_workers=2) as pool:
    text_future = pool.submit(text_index.search, text_encoder.embed([query])[0], 3)
    image_future = pool.submit(retrieve_b_color, query, 3)
    concurrent_hits = rrf([text_future.result(), image_future.result()])[:3]
print("Same IDs as sequential:", [h["id"] for h in concurrent_hits] == [h["id"] for h in retrieve_c(query)])

Same IDs as sequential: True


![Recall, hit rate, reciprocal rank, and answer correctness measure different stages.](assets/11_evaluation.svg)

## Step 4 · Name the metric precisely

Suppose a question has two relevant sources, and the top three results contain just
one of them at rank 2. **Recall@3 = 1/2**, **Hit@3 = 1**, and **reciprocal rank@3 =
1/2**. Across questions, average Hit gives hit rate and average reciprocal rank gives
MRR. Our MRR is explicitly truncated at `k`: a relevant item below `k` receives zero.

The reference notebook's cell 37 calls an “any relevant result?” calculation recall.
That is hit rate; it equals recall only when every query has exactly one relevant
source. Our gold set deliberately includes multiple relevant sources.

In [7]:
one_question = [{"query": "worked example", "relevant": {"A", "B"}}]
worked_retrieval = lambda query, k: [toy_hit("X", .9), toy_hit("A", .8), toy_hit("Y", .7)][:k]
summary, detail = evaluate(worked_retrieval, one_question, k=3)
display(summary)
display(detail)
assert summary["Recall@3"] == .5 and summary["HitRate@3"] == 1.0 and summary["MRR@3"] == .5

{'k': 3,
 'supported_queries': 1,
 'unsupported_queries': 0,
 'Recall@3': 0.5,
 'HitRate@3': 1.0,
 'MRR@3': 0.5,
 'unsupported_evidence_return_rate': None}

[{'query': 'worked example',
  'retrieved': ['X', 'A', 'Y'],
  'relevant': ['A', 'B'],
  'recall': 0.5,
  'hit': 1,
  'rr': 0.5,
  'unsupported_return': None}]

## Step 5 · Evaluate the actual teaching baselines

The gold set defines sources useful for answering each question. Some questions need
one chart; others accept both a chart and a contextual memo. Two queries have no
supporting source: one uses unseen words; the other asks about a future forecast
using familiar solar-energy words. The second is intentionally harder to reject.

Recall is undefined when there are no relevant sources, so we exclude those queries
from recall/hit/MRR averages and report **unsupported evidence-return rate** separately.
Returning a candidate is not the same as generating a false answer. Answer-level
refusal and groundedness need a separate evaluation.

In [8]:
print("Question labels:")
display([{**case, "relevant": sorted(case["relevant"])} for case in GOLD])
strategies = {"A: lexical + reference descriptions": retrieve_a,
              "B teaching baseline: color-only images": retrieve_b_color,
              "C teaching baseline: lexical memos + color RRF": retrieve_c}
baseline_results = {}
for name, retrieve in strategies.items():
    baseline_results[name] = evaluate(retrieve, k=3)
display([{ "strategy": name, **result[0]} for name, result in baseline_results.items()])

Question labels:


[{'query': 'monthly solar energy',
  'relevant': ['chart_energy', 'memo_schedule'],
  'kind': 'multiple sources'},
 {'query': 'purple panel temperature chart',
  'relevant': ['chart_temperature'],
  'kind': 'visual and text'},
 {'query': 'green water cleaning methods',
  'relevant': ['chart_water', 'memo_cleaning'],
  'kind': 'multiple sources'},
 {'query': 'controller E17 warning',
  'relevant': ['memo_safety', 'screenshot_controller'],
  'kind': 'exact code and context'},
 {'query': 'Thursday classroom review',
  'relevant': ['memo_schedule'],
  'kind': 'text'},
 {'query': 'orange bar chart', 'relevant': ['chart_energy'], 'kind': 'visual'},
 {'query': 'lunar greenhouse harvest',
  'relevant': [],
  'kind': 'unsupported, out of vocabulary'},
 {'query': 'solar energy next winter forecast',
  'relevant': [],
  'kind': 'unsupported, overlapping vocabulary'}]

[{'strategy': 'A: lexical + reference descriptions',
  'k': 3,
  'supported_queries': 6,
  'unsupported_queries': 2,
  'Recall@3': 1.0,
  'HitRate@3': 1.0,
  'MRR@3': 1.0,
  'unsupported_evidence_return_rate': 0.5},
 {'strategy': 'B teaching baseline: color-only images',
  'k': 3,
  'supported_queries': 6,
  'unsupported_queries': 2,
  'Recall@3': 0.4166666666666667,
  'HitRate@3': 0.5,
  'MRR@3': 0.5,
  'unsupported_evidence_return_rate': 0.0},
 {'strategy': 'C teaching baseline: lexical memos + color RRF',
  'k': 3,
  'supported_queries': 6,
  'unsupported_queries': 2,
  'Recall@3': 0.8333333333333334,
  'HitRate@3': 1.0,
  'MRR@3': 1.0,
  'unsupported_evidence_return_rate': 0.5}]

Do not stop at an average. The next cell prints each query for C, including the
retrieved sources and labelled relevance. Look for a query whose color branch had no
direction, a memo-only question, and an unsupported question that still returned hits.
Those failures explain *why* a score changed.

In [9]:
display(baseline_results["C teaching baseline: lexical memos + color RRF"][1])
print("Overlap does not imply answerability:")
display(show_hits(retrieve_a("solar energy next winter forecast", 3)))

[{'query': 'monthly solar energy',
  'retrieved': ['memo_schedule', 'memo_safety', 'memo_cleaning'],
  'relevant': ['chart_energy', 'memo_schedule'],
  'recall': 0.5,
  'hit': 1,
  'rr': 1.0,
  'unsupported_return': None},
 {'query': 'purple panel temperature chart',
  'retrieved': ['chart_temperature', 'memo_cleaning', 'chart_energy'],
  'relevant': ['chart_temperature'],
  'recall': 1.0,
  'hit': 1,
  'rr': 1.0,
  'unsupported_return': None},
 {'query': 'green water cleaning methods',
  'retrieved': ['chart_water', 'memo_cleaning', 'chart_temperature'],
  'relevant': ['chart_water', 'memo_cleaning'],
  'recall': 1.0,
  'hit': 1,
  'rr': 1.0,
  'unsupported_return': None},
 {'query': 'controller E17 warning',
  'retrieved': ['memo_safety', 'memo_cleaning', 'memo_schedule'],
  'relevant': ['memo_safety', 'screenshot_controller'],
  'recall': 0.5,
  'hit': 1,
  'rr': 1.0,
  'unsupported_return': None},
 {'query': 'Thursday classroom review',
  'retrieved': ['memo_schedule', 'memo_cleani

Overlap does not imply answerability:


[{'rank': 1,
  'id': 'memo_schedule',
  'source': 'memo_schedule',
  'modality': 'text',
  'score': 0.5392},
 {'rank': 2,
  'id': 'chart_energy',
  'source': 'chart_energy',
  'modality': 'image',
  'score': 0.3638},
 {'rank': 3,
  'id': 'memo_safety',
  'source': 'memo_safety',
  'modality': 'text',
  'score': 0.2425}]

Search almost always finds the nearest item when a nonzero query is present. A
forecast question can retrieve a historical chart without the chart containing any
forecast. A similarity threshold might reject some such cases, but its value must
be chosen on labelled development data; a universal threshold like `0.7` has no
general meaning across encoders and corpora.

For a real study, separate development queries from held-out test queries, report
sample sizes, inspect relevance disagreements, and measure retrieval separately from
answer correctness. Eight classroom queries illustrate mechanics; they do not establish
a reliable ranking of commercial systems.

## Step 6 · Optional genuine A/B/C comparison

With `MM_RUN_MODELS=1`, this cell runs real local FastEmbed models on the same questions:
**A** uses BGE text embeddings over memos and reference descriptions; **B** uses one
joint CLIP collection; **C** fuses BGE memo ranks with CLIP image ranks. All three use
actual Qdrant retrieval. Results are computed, not copied from the color baseline.

This comparison still gives A hand-authored captions. Report that advantage; repeat
with measured live parses if you want to study parser errors. The local-model flag
does not enable paid vision calls. Live parsing additionally requires `MM_RUN_LIVE=1`.

In [10]:
if RUN_MODELS:
    from live import run_model_comparison
    real_results = run_model_comparison(k=3)
    display([{ "strategy": name, **result[0]} for name, result in real_results.items()])
    for name, (_, per_query) in real_results.items():
        print(name)
        display(per_query)
else:
    print("Real A/B/C comparison skipped. No CLIP or BGE metric is fabricated.")

Real A/B/C comparison skipped. No CLIP or BGE metric is fabricated.


## Step 7 · Study the trade-off, not just the winner

Larger `k` usually improves the chance of including useful evidence, but passes more
irrelevant material to a reader and can increase latency or cost. A useful retrieval
experiment measures a small curve, not a single convenient cutoff. The next cell
changes `k` while keeping questions and strategy fixed. A second experiment should
hold final `k=3` fixed and vary C's per-branch pool (for example 1, 2, then 3), recording
the number of candidates each searcher examines. Fusion cannot recover a useful
source omitted from every branch's candidate pool.

In [11]:
display([evaluate(retrieve_a, k=k)[0] for k in (1, 2, 3)])

[{'k': 1,
  'supported_queries': 6,
  'unsupported_queries': 2,
  'Recall@1': 0.75,
  'HitRate@1': 1.0,
  'MRR@1': 1.0,
  'unsupported_evidence_return_rate': 0.5},
 {'k': 2,
  'supported_queries': 6,
  'unsupported_queries': 2,
  'Recall@2': 1.0,
  'HitRate@2': 1.0,
  'MRR@2': 1.0,
  'unsupported_evidence_return_rate': 0.5},
 {'k': 3,
  'supported_queries': 6,
  'unsupported_queries': 2,
  'Recall@3': 1.0,
  'HitRate@3': 1.0,
  'MRR@3': 1.0,
  'unsupported_evidence_return_rate': 0.5}]

### Exercise · A fairer study

Add two held-out questions: one with multiple relevant sources and one unsupported
question that shares vocabulary with the corpus. Label them by inspecting the actual
sources before looking at retrieval output. Then describe how you would compare
reference captions with live model captions without changing everything at once.

<details><summary>Worked design</summary>

Hold the corpus, query set, encoder, `k`, and relevance labels fixed. Change only
caption source. Measure parse label/value correctness first, then retrieval and answer
quality. Keep development questions separate from final test questions. Do not tune a
threshold on the same small set used to claim the final result.
</details>

Next: [05 · PDF pages and video moments](05_pdf_and_video.ipynb).
Source connections: reference cells 30–31 (RRF), 37–38 (evaluation), and 43–44
(video source identities). Our implementation prevents duplicate frames from gaining
repeated votes inside one ranked list.